# EatHalf: corrected reproducible classification workflow

**Oludotun Babayemi | Research prototype**

This notebook calls the repository's Python modules so notebook and CLI use the same preprocessing and evaluation logic. Start with Python 3.11 and install `requirements.txt` from the repository root. Select that environment as the notebook kernel.

The original notebook's 83.97% EfficientNetB0 test accuracy is historical and has not been reproduced by this workflow. Running this notebook trains new models. No trained EatHalf weights are included.

Data: 17 selected Food-101 categories. A custom 70/15/15 split is used, not the official benchmark. Set your data path below. Images and artifacts remain local and ignored by Git.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "eathalf").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "eathalf").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))

DATA_DIR = Path("C:/datasets/Food_101")  # EDIT: must contain the 17 class folders
RUN_DIR = ROOT / "artifacts" / "run-001"  # choose a fresh directory per experiment
SEED = 42
MODELS = ["efficientnet"]
# For the full comparison, decide before evaluating test:
# MODELS = ["cnn", "cnn_aug", "mobilenet128", "mobilenet224", "efficientnet", "resnet"]
EPOCHS = 10
BATCH_SIZE = 32


## 1. Validate images and prepare manifests

Checks all supported images, removes exact decoded-pixel duplicates, rejects conflicting duplicate labels, and saves a stratified split. Existing run directories are rejected to avoid mixing runs. Near-duplicates are not detected. A new full Food-101 download may have more apple-pie images than the historical copy.


In [ ]:
from eathalf.data import prepare
import json

prepare(DATA_DIR, RUN_DIR, seed=SEED)
summary = json.loads((RUN_DIR / "split_summary.json").read_text())
summary


In [ ]:
import matplotlib.pyplot as plt

counts = summary["class_counts"]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(counts), list(counts.values()))
ax.set_xlabel("Valid unique images")
ax.set_title("EatHalf source class distribution after validation")
plt.tight_layout()
plt.show()


## 2. Train candidates and select on validation

Preprocessing is embedded in saved models. Labels and predictions are collected from the same evaluation batches. Each candidate retains its best validation-loss checkpoint; model selection uses validation macro F1 to give each class equal weight. A GPU is recommended for comparing all six candidates. Failed or completed training runs are preserved: use a fresh run directory to restart.


In [ ]:
from eathalf.workflow import train

train(DATA_DIR, RUN_DIR, names=MODELS, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
comparison = json.loads((RUN_DIR / "validation_comparison.json").read_text())
comparison


## 3. Evaluate the selected model on test

Run after deciding the candidate set and selection rule. The command refuses to overwrite existing test metrics. Do not change the model after reading test results and report the same test set as untouched. Images inspected in the original experiment remain previously seen data; independent external images are needed for a stronger final assessment.


In [ ]:
from eathalf.workflow import evaluate_test

evaluate_test(DATA_DIR, RUN_DIR)
results = json.loads((RUN_DIR / "test_metrics.json").read_text())
{k: results[k] for k in ("model", "n", "accuracy", "macro_f1", "weighted_f1")}


In [ ]:
from eathalf.data import CLASSES
import numpy as np

fig, ax = plt.subplots(figsize=(12, 10))
plot = ax.imshow(np.array(results["confusion_matrix"]), cmap="Blues")
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=90)
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Selected classifier: custom test-set confusion matrix")
fig.colorbar(plot, ax=ax)
plt.tight_layout()
plt.show()


## 4. Local inference and optional detection

From a VS Code terminal in the repository root:

```powershell
python -m eathalf.predict --image "C:/photos/meal.jpg" --run-dir artifacts/run-001
```

Optional exploration:

```powershell
python -m pip install -r requirements-detection.txt
python -m eathalf.detect --image "C:/photos/meal.jpg"
```

YOLO uses pretrained COCO categories and supports only some foods. Its outputs are separate from the classifier's test evaluation. Detector confidence is not accuracy; image class labels cannot establish bounding-box quality.

## Interpretation

These experiments measure whole-image classification of supported foods. They do not estimate calories, portion weight, ingredients, allergens, or health effects. Softmax probabilities are uncalibrated. Check per-class metrics, inspect errors, and validate on external images before product claims. See the README, model card, data provenance, and third-party notices.
